<a href="https://colab.research.google.com/github/Risshhaayy/Risshhaayy/blob/main/qryptoD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests

ip = requests.get("https://api.ipify.org").text
print("Colab Public IP:", ip)



Colab Public IP: 34.50.189.107


In [ ]:
#!/usr/bin/env python3
"""
╔══════════════════════════════════════════════════════════════╗
║        BTCUSD ALGO TRADING BOT — EMA CROSSOVER + TP/SL      ║
║                                                              ║
║  Strategy : EMA 8/33 Pure Crossover on 1-min candles         ║
║  Broker   : Delta Exchange Testnet (Paper Trading)           ║
║  Asset    : BTCUSD Perpetual                                 ║
║  TP / SL  : +3% Take Profit / -1% Stop Loss                 ║
║  Risk     : 2.5% max daily drawdown auto-halt                ║
╚══════════════════════════════════════════════════════════════╝
"""


# ═══════════════════════════════════════════════════════════════
# STEP 1 — INSTALL & IMPORT DEPENDENCIES
# ═══════════════════════════════════════════════════════════════

import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "delta-rest-client", "pandas", "numpy", "ta", "requests"])

import time
import requests as http_requests
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

import pandas as pd
import ta

from delta_rest_client import DeltaRestClient, OrderType, TimeInForce

print("✅ All dependencies loaded!")


# ═══════════════════════════════════════════════════════════════
# STEP 2 — API KEYS & CLIENT SETUP (DELTA EXCHANGE TESTNET)
# ═══════════════════════════════════════════════════════════════
# Delta Exchange India Testnet
# Dashboard: https://testnet.delta.exchange/

BASE_URL   = "https://cdn-ind.testnet.deltaex.org"
API_KEY    = "uJhfmoJFXEIhjUc5AVAmkGTiuS2rVy"
API_SECRET = "I5opzAtrJuZfmSZo2WQiVzb0rBDaPl4sZOnFuPq3fbDkT4aBumcqIPCzvOz9"

delta_client = DeltaRestClient(
    base_url=BASE_URL,
    api_key=API_KEY,
    api_secret=API_SECRET,
)

# ── Resolve BTCUSD product ID ──
def get_product_id(symbol="BTCUSD"):
    """Fetch the product_id for the given symbol from Delta Exchange."""
    try:
        ticker = delta_client.get_ticker(symbol)
        return ticker['product_id']
    except Exception:
        pass
    # Fallback: search products list
    try:
        response = delta_client.request("GET", "/v2/products", auth=False)
        products = response.json().get("result", [])
        for p in products:
            if p.get("symbol") == symbol:
                return p["id"]
    except Exception as e:
        print(f"⚠️  Could not resolve product_id: {e}")
    return None

PRODUCT_SYMBOL = "BTCUSD"
PRODUCT_ID = get_product_id(PRODUCT_SYMBOL)

if PRODUCT_ID:
    print(f"✅ Product resolved: {PRODUCT_SYMBOL} → ID {PRODUCT_ID}")
else:
    print("⚠️  Could not resolve product ID — will use symbol-based orders")

# ── Verify connection ──
try:
    balances = delta_client.request("GET", "/v2/wallet/balances", auth=True)
    bal_data = balances.json()
    if bal_data.get("success"):
        wallets = bal_data.get("result", [])
        for w in wallets:
            if w.get("asset_symbol") in ("USDT", "BTC", "USD"):
                print(f"  💰 {w['asset_symbol']}: {w.get('balance', 'N/A')}")
    print("=" * 60)
    print("📊 DELTA EXCHANGE TESTNET CONNECTED")
    print("=" * 60)
except Exception as e:
    print(f"⚠️  Connection check: {e}")
    print("=" * 60)
    print("📊 Proceeding — connection will be verified on first trade")
    print("=" * 60)


# ═══════════════════════════════════════════════════════════════
# STEP 3 — STRATEGY CONFIGURATION
# ═══════════════════════════════════════════════════════════════

SYMBOL        = "BTCUSD"        # Delta product symbol
EMA_SHORT     = 8               # Fast EMA period
EMA_LONG_     = 33              # Slow EMA period
POSITION_SIZE = 10              # Contracts per trade (Delta uses contracts)
TP_PERCENT    = 0.03            # Take Profit = 3%
SL_PERCENT    = 0.01            # Stop Loss = 1%
MAX_DAILY_DD  = 0.025           # Max 2.5% daily drawdown before halt
POLL_INTERVAL = 20              # Seconds between each scan cycle (was 60)


# ═══════════════════════════════════════════════════════════════
# STEP 4 — DATA FETCHING & INDICATOR CALCULATION
# ═══════════════════════════════════════════════════════════════
# Fetches 5-min OHLCV candles from Delta Exchange REST API,
# and computes EMA 8 / EMA 33 on the close price.

def fetch_candles(symbol=SYMBOL, lookback_minutes=500):
    """
    Fetch 1-min candles from Delta Exchange → compute EMA 8/33.
    Uses GET /v2/history/candles endpoint.
    Returns a DataFrame with columns: open, high, low, close, volume, ema_8, ema_33
    """
    now   = int(datetime.now(ZoneInfo("UTC")).timestamp())
    start = now - (lookback_minutes * 60)

    url = f"{BASE_URL}/v2/history/candles"
    params = {
        "resolution": "1m",
        "symbol": symbol,
        "start": str(start),
        "end": str(now),
    }

    try:
        resp = http_requests.get(url, params=params, timeout=10)
        resp.raise_for_status()
        data = resp.json()
    except Exception as e:
        print(f"⚠️  Candle fetch error: {e}")
        return pd.DataFrame()

    if not data.get("success") or not data.get("result"):
        print("⚠️  No candle data returned.")
        return pd.DataFrame()

    candles = data["result"]
    records = [{
        "timestamp": pd.to_datetime(c["time"], unit="s", utc=True),
        "open":   float(c["open"]),
        "high":   float(c["high"]),
        "low":    float(c["low"]),
        "close":  float(c["close"]),
        "volume": float(c["volume"]),
    } for c in candles]

    df = pd.DataFrame(records)
    if df.empty:
        print("⚠️  Empty DataFrame after parsing.")
        return df

    df.set_index("timestamp", inplace=True)
    df.sort_index(inplace=True)

    # Compute EMA indicators
    df["ema_8"]  = ta.trend.ema_indicator(df["close"], window=EMA_SHORT)
    df["ema_33"] = ta.trend.ema_indicator(df["close"], window=EMA_LONG_)
    df.dropna(inplace=True)

    if not df.empty:
        last = df.iloc[-1]
        print(f"📊 Candle: {df.index[-1]}  Close: ${last['close']:,.2f}  "
              f"EMA8: ${last['ema_8']:,.2f}  EMA33: ${last['ema_33']:,.2f}")
    return df


# Quick sanity check
print("\n🔍 Fetching initial candle data...")
test_df = fetch_candles()
if not test_df.empty:
    print(f"\n✅ {len(test_df)} bars loaded")
    print(test_df[["close", "ema_8", "ema_33"]].tail())
else:
    print("⚠️  No bars loaded — bot will retry in main loop")


# ═══════════════════════════════════════════════════════════════
# STEP 5 — CROSSOVER SIGNAL DETECTION
# ═══════════════════════════════════════════════════════════════
# Detects EMA 8/33 crossovers:
#   • EMA8 crosses ABOVE EMA33  → LONG signal
#   • EMA8 crosses BELOW EMA33  → SHORT signal

last_crossover = None


def detect_crossover(df):
    """
    Compare the last two candles to detect a fresh crossover.
    Returns 'LONG', 'SHORT', or None.
    """
    global last_crossover
    if len(df) < 2:
        return None

    prev, curr = df.iloc[-2], df.iloc[-1]

    ema8_above      = curr["ema_8"] > curr["ema_33"]
    prev_ema8_above = prev["ema_8"] > prev["ema_33"]

    cross_up   = not prev_ema8_above and ema8_above
    cross_down = prev_ema8_above and not ema8_above

    signal = None
    if cross_up:
        signal = "LONG"
    elif cross_down:
        signal = "SHORT"

    # Initialize state on first run
    if signal is None and last_crossover is None:
        last_crossover = "LONG" if ema8_above else "SHORT"
        print(f"📌 Initial: EMA8 {'>' if ema8_above else '<'} EMA33")
        return None

    # Only fire if it's a NEW crossover
    if signal and signal != last_crossover:
        last_crossover = signal
        print(f"🔔 CROSSOVER: {signal}  |  "
              f"EMA8=${curr['ema_8']:,.2f}  EMA33=${curr['ema_33']:,.2f}")
        return signal

    return None


print("✅ Signal detection ready.")


# ═══════════════════════════════════════════════════════════════
# STEP 6 — POSITION TRACKING (DELTA EXCHANGE)
# ═══════════════════════════════════════════════════════════════
# Checks Delta Exchange for any open BTCUSD position.
# Delta positions: size > 0 means open, entry_price is avg entry.

def get_open_position():
    """Returns a dict with position details, or None if flat."""
    try:
        if PRODUCT_ID:
            pos = delta_client.get_position(PRODUCT_ID)
        else:
            # Fallback: use margined positions
            resp = delta_client.request("GET", "/v2/positions/margined", auth=True)
            positions = resp.json().get("result", [])
            pos = None
            for p in positions:
                if p.get("product_symbol") == PRODUCT_SYMBOL:
                    pos = p
                    break
            if pos is None:
                return None

        size = int(pos.get("size", 0))
        if size == 0:
            return None

        entry_price = float(pos.get("entry_price", 0))
        # Determine side from size: positive = long, negative = short
        # Delta uses positive size for both, side indicated separately
        # or we use our tracked side
        return {
            "symbol":        PRODUCT_SYMBOL,
            "size":          abs(size),
            "entry_price":   entry_price,
            "realized_pnl":  pos.get("realized_pnl", "0"),
            "product_id":    pos.get("product_id", PRODUCT_ID),
        }
    except Exception as e:
        print(f"⚠️  Position check error: {e}")
    return None


pos = get_open_position()
print(f"Position: {pos if pos else 'None (flat — ready to trade)'}")


# ═══════════════════════════════════════════════════════════════
# STEP 7 — ORDER EXECUTION WITH TP/SL BRACKET ORDERS
# ═══════════════════════════════════════════════════════════════
# • open_position()  — Opens LONG or SHORT + places TP/SL bracket
# • close_position() — Closes the current position
# • flip_position()  — Closes current + Opens opposite direction
# • place_tp_sl()    — Places bracket (TP +3%, SL -1%)

current_trade_side  = None   # "LONG" | "SHORT" | None
current_entry_price = None   # Track entry price for TP/SL monitoring


def place_tp_sl(side, entry_price):
    """
    Place a bracket order (TP + SL) after position entry.
    TP = +3% from entry, SL = -1% from entry.
    Uses Delta Exchange bracket order API.
    """
    if side == "LONG":
        tp_price = round(entry_price * (1 + TP_PERCENT), 1)
        sl_price = round(entry_price * (1 - SL_PERCENT), 1)
    else:  # SHORT
        tp_price = round(entry_price * (1 - TP_PERCENT), 1)
        sl_price = round(entry_price * (1 + SL_PERCENT), 1)

    print(f"🎯 TP: ${tp_price:,.1f} ({TP_PERCENT*100}%)  |  "
          f"🛑 SL: ${sl_price:,.1f} ({SL_PERCENT*100}%)")

    bracket_payload = {
        "product_id": PRODUCT_ID,
        "product_symbol": PRODUCT_SYMBOL,
        "stop_loss_order": {
            "order_type": "market_order",
            "stop_price": str(int(sl_price)),
        },
        "take_profit_order": {
            "order_type": "market_order",
            "stop_price": str(int(tp_price)),
        },
        "bracket_stop_trigger_method": "last_traded_price",
    }

    try:
        resp = delta_client.request(
            "POST", "/v2/orders/bracket",
            payload=bracket_payload,
            auth=True
        )
        result = resp.json()
        if result.get("success"):
            print(f"✅ Bracket order placed (TP/SL)")
        else:
            print(f"⚠️  Bracket response: {result}")
            # Fallback: place individual stop orders
            place_individual_tp_sl(side, entry_price, tp_price, sl_price)
    except Exception as e:
        print(f"⚠️  Bracket order error: {e}")
        # Fallback: place as individual stop orders
        place_individual_tp_sl(side, entry_price, tp_price, sl_price)


def place_individual_tp_sl(side, entry_price, tp_price, sl_price):
    """Fallback: place TP and SL as individual stop orders."""
    try:
        # Stop Loss order
        sl_side = "sell" if side == "LONG" else "buy"
        delta_client.place_stop_order(
            product_id=PRODUCT_ID,
            size=POSITION_SIZE,
            side=sl_side,
            stop_price=str(int(sl_price)),
            order_type=OrderType.MARKET,
        )
        print(f"  ✅ SL order placed at ${sl_price:,.1f}")
    except Exception as e:
        print(f"  ❌ SL order failed: {e}")

    try:
        # Take Profit order (as stop order with opposite trigger)
        tp_side = "sell" if side == "LONG" else "buy"
        tp_order = {
            "product_id": PRODUCT_ID,
            "product_symbol": PRODUCT_SYMBOL,
            "size": POSITION_SIZE,
            "side": tp_side,
            "order_type": "market_order",
            "stop_order_type": "take_profit_order",
            "stop_price": str(int(tp_price)),
            "stop_trigger_method": "last_traded_price",
            "reduce_only": True,
        }
        delta_client.create_order(tp_order)
        print(f"  ✅ TP order placed at ${tp_price:,.1f}")
    except Exception as e:
        print(f"  ❌ TP order failed: {e}")


def cancel_all_open_orders():
    """Cancel all open/pending orders for the product."""
    try:
        payload = {"product_id": PRODUCT_ID}
        delta_client.request("DELETE", "/v2/orders/all", payload=payload, auth=True)
        print("🧹 All open orders cancelled.")
    except Exception as e:
        print(f"⚠️  Cancel orders: {e}")


def open_position(side, price):
    """Submit a market order to open a new position + place TP/SL."""
    global current_trade_side, current_entry_price
    order_side = "buy" if side == "LONG" else "sell"

    print(f"\n{'=' * 55}")
    print(f"📤 OPEN {side}  |  {POSITION_SIZE} contracts  |  ~${price:,.2f}")
    print(f"{'=' * 55}")

    try:
        order = delta_client.place_order(
            product_id=PRODUCT_ID,
            size=POSITION_SIZE,
            side=order_side,
            order_type=OrderType.MARKET,
        )
        current_trade_side = side
        current_entry_price = price
        print(f"✅ Order submitted — ID: {order.get('id', 'N/A')}")

        # Wait briefly for fill
        time.sleep(2)

        # Check actual entry price from position
        pos = get_open_position()
        if pos and pos["entry_price"] > 0:
            current_entry_price = pos["entry_price"]
            print(f"📌 Actual entry price: ${current_entry_price:,.2f}")

        # Place TP/SL bracket order
        place_tp_sl(side, current_entry_price)

        return order
    except Exception as e:
        print(f"❌ FAILED: {e}")
        return None


def close_position():
    """Close the currently open position."""
    global current_trade_side, current_entry_price
    pos = get_open_position()
    if pos is None:
        current_trade_side = None
        current_entry_price = None
        return

    close_side = "sell" if current_trade_side == "LONG" else "buy"
    print(f"📥 CLOSE {current_trade_side}  |  Entry: ${current_entry_price or 0:,.2f}")

    # Cancel any pending TP/SL orders first
    cancel_all_open_orders()

    try:
        order = delta_client.place_order(
            product_id=PRODUCT_ID,
            size=pos["size"],
            side=close_side,
            order_type=OrderType.MARKET,
            reduce_only='true',
        )
        print(f"✅ Closed — ID: {order.get('id', 'N/A')}")
        current_trade_side = None
        current_entry_price = None
    except Exception as e:
        print(f"❌ Close failed: {e}")


def flip_position(new_side, price):
    """Close current position and immediately open the opposite side."""
    print(f"\n🔄 FLIP: {current_trade_side} → {new_side}")
    close_position()
    time.sleep(2)
    open_position(new_side, price)


def check_tp_sl_hit():
    """
    Check if TP or SL was hit (position closed by bracket order).
    Returns True if the position was closed, False if still open.
    """
    global current_trade_side, current_entry_price

    if current_trade_side is None:
        return False  # No active trade

    pos = get_open_position()
    if pos is None:
        # Position was closed — TP or SL was hit!
        print(f"\n🏁 Trade completed — TP/SL hit! (was {current_trade_side})")
        current_trade_side = None
        current_entry_price = None
        return True

    # Position still open — check unrealized P&L manually
    if current_entry_price and current_entry_price > 0:
        try:
            ticker = delta_client.get_ticker(PRODUCT_SYMBOL)
            mark_price = float(ticker.get("mark_price", 0))
            if mark_price > 0:
                if current_trade_side == "LONG":
                    pnl_pct = (mark_price - current_entry_price) / current_entry_price
                else:
                    pnl_pct = (current_entry_price - mark_price) / current_entry_price
                print(f"  📊 Mark: ${mark_price:,.2f}  P&L: {pnl_pct*100:+.2f}%")
        except Exception:
            pass

    return False


print("✅ Order execution ready.")


# ═══════════════════════════════════════════════════════════════
# STEP 8 — RISK MANAGEMENT (DAILY DRAWDOWN GUARD)
# ═══════════════════════════════════════════════════════════════
# Tracks daily equity via wallet balance. If drawdown exceeds
# MAX_DAILY_DD (2.5%), halts all trading for the rest of the day.

daily_start_equity = None
trading_halted     = False


def get_wallet_balance():
    """Get total wallet balance from Delta Exchange."""
    try:
        resp = delta_client.request("GET", "/v2/wallet/balances", auth=True)
        data = resp.json()
        if data.get("success"):
            total = 0
            for w in data.get("result", []):
                bal = float(w.get("balance", 0))
                total += bal
            return total if total > 0 else None
    except Exception as e:
        print(f"⚠️  Balance check: {e}")
    return None


def check_daily_drawdown():
    """Returns True if drawdown limit hit (trading should halt)."""
    global daily_start_equity, trading_halted
    equity = get_wallet_balance()

    if equity is None:
        return False  # Can't check, continue trading

    if daily_start_equity is None:
        daily_start_equity = equity
        print(f"📌 Daily start equity: {equity:,.4f}")
        return False

    if daily_start_equity == 0:
        return False

    dd = (daily_start_equity - equity) / daily_start_equity
    if dd >= MAX_DAILY_DD:
        print(f"🚨 DRAWDOWN {dd * 100:.2f}% — TRADING HALTED")
        trading_halted = True
        return True

    print(f"📉 Daily P&L: {-dd * 100:+.2f}%")
    return False


def reset_daily_tracker():
    """Reset at the start of each new UTC day."""
    global daily_start_equity, trading_halted
    daily_start_equity = None
    trading_halted = False
    print("🔄 Daily tracker reset.")


print("✅ Risk management ready.")


# ═══════════════════════════════════════════════════════════════
# STEP 9 — STATUS LOGGING (PRETTY DASHBOARD)
# ═══════════════════════════════════════════════════════════════

def log_status(df, position, signal):
    """Print a formatted status dashboard to the terminal."""
    if df.empty:
        return

    c   = df.iloc[-1]
    ts  = str(df.index[-1])[:19]
    now = datetime.now(ZoneInfo("UTC")).strftime("%H:%M:%S UTC")

    trend = "📈 BULL" if c["ema_8"] > c["ema_33"] else "📉 BEAR"
    gap   = abs(c["ema_8"] - c["ema_33"])

    print(f"\n┌───────────────────────────────────────────────────┐")
    print(f"│ 🕐 {now}  Candle: {ts}              │")
    print(f"│ Price: ${c['close']:,.2f}  {trend}  Gap: ${gap:,.2f}     │")
    print(f"│ EMA8: ${c['ema_8']:,.2f}   EMA33: ${c['ema_33']:,.2f}          │")

    if position:
        entry = position.get("entry_price", 0)
        print(f"│ 📌 {current_trade_side or 'N/A'} {position['size']} contracts  "
              f"Entry: ${entry:,.2f}  │")
        if current_entry_price and current_entry_price > 0:
            if current_trade_side == "LONG":
                tp = current_entry_price * (1 + TP_PERCENT)
                sl = current_entry_price * (1 - SL_PERCENT)
            else:
                tp = current_entry_price * (1 - TP_PERCENT)
                sl = current_entry_price * (1 + SL_PERCENT)
            print(f"│ 🎯 TP: ${tp:,.1f}  🛑 SL: ${sl:,.1f}                  │")
    elif signal:
        print(f"│ 🔔 {signal:<46} │")
    else:
        print(f"│ ⏳ Waiting for crossover...                       │")

    print(f"└───────────────────────────────────────────────────┘")


print("✅ Logging ready.")


# ═══════════════════════════════════════════════════════════════
# STEP 10 — MAIN TRADING LOOP
# ═══════════════════════════════════════════════════════════════
# Infinite loop:
#   1. Reset daily tracker at midnight UTC
#   2. Check daily drawdown → halt if exceeded
#   3. Check if TP/SL was hit → reset for next trade
#   4. Fetch candles + compute EMAs
#   5. Detect crossover signal
#   6. Execute trade (open / flip / hold)
#   7. Log status dashboard
#   8. Sleep 20s and repeat

# Reset global state for main loop
last_day           = None
last_crossover     = None
current_trade_side = None
current_entry_price = None

print("\n" + "🚀" * 25)
print("   BTCUSD BOT — Delta Exchange Testnet")
print("   Strategy: EMA 8/33 Pure Crossover")
print("   TP: +3%  |  SL: -1%")
print("   Entry: Crossover → Open + Bracket TP/SL")
print("   Exit : TP/SL hit or Opposite Crossover → Flip")
print(f"   Poll: every {POLL_INTERVAL}s")
print("🚀" * 25 + "\n")

cycle = 0

while True:
    cycle += 1
    try:
        now   = datetime.now(ZoneInfo("UTC"))
        today = now.date()

        # ── Daily reset at midnight UTC ──
        if last_day is None or today != last_day:
            reset_daily_tracker()
            last_day = today

        print(f"\n{'─' * 55}")
        print(f"🔄 Cycle #{cycle} — {now.strftime('%Y-%m-%d %H:%M:%S UTC')}")
        print(f"{'─' * 55}")

        # ── Check drawdown guard ──
        if check_daily_drawdown():
            print("⏸️  Trading halted for today.")
            time.sleep(POLL_INTERVAL)
            continue

        # ── Check if TP/SL was hit (trade completed) ──
        if current_trade_side is not None:
            if check_tp_sl_hit():
                print("🔄 Trade completed! Scanning for next crossover...")
                # Continue to next cycle to find new signal
                time.sleep(POLL_INTERVAL)
                continue

        # ── Fetch candles & compute EMAs ──
        df = fetch_candles()
        if df.empty:
            print("⏳ No data — retrying next cycle...")
            time.sleep(POLL_INTERVAL)
            continue

        # ── Check current position ──
        position = get_open_position()

        # Sync state if position was closed externally
        if position is None and current_trade_side is not None:
            print("ℹ️  Position closed externally — resetting state.")
            current_trade_side = None
            current_entry_price = None

        # ── Detect crossover signal ──
        signal = detect_crossover(df)

        # ── Execute trade logic ──
        if signal:
            price = df.iloc[-1]["close"]

            if position is None:
                # No position → open new one with TP/SL
                open_position(signal, price)
            elif current_trade_side and signal != current_trade_side:
                # Opposite signal → flip position
                flip_position(signal, price)
            else:
                # Same direction → hold
                print(f"ℹ️  Already {current_trade_side} — holding.")
        else:
            if position:
                print(f"📌 Holding {current_trade_side}")
            else:
                print("🔍 No crossover — scanning...")

        # ── Log status dashboard ──
        position = get_open_position()
        log_status(df, position, signal)

    except KeyboardInterrupt:
        print("\n🛑 Bot stopped by user.")
        break
    except Exception as e:
        print(f"❌ Error in cycle #{cycle}: {e}")
        import traceback
        traceback.print_exc()

    print(f"\n⏰ Next scan in {POLL_INTERVAL}s...")
    time.sleep(POLL_INTERVAL)


✅ All dependencies loaded!
✅ Product resolved: BTCUSD → ID 84
  💰 BTC: 0.25
  💰 USD: 1000
📊 DELTA EXCHANGE TESTNET CONNECTED

🔍 Fetching initial candle data...
📊 Candle: 2026-02-26 08:46:00+00:00  Close: $67,996.50  EMA8: $67,985.36  EMA33: $67,998.39

✅ 467 bars loaded
                             close         ema_8        ema_33
timestamp                                                     
2026-02-26 08:42:00+00:00  67965.8  68015.212058  68008.536810
2026-02-26 08:43:00+00:00  67903.7  67990.431601  68002.369939
2026-02-26 08:44:00+00:00  67957.4  67983.091245  67999.724649
2026-02-26 08:45:00+00:00  67979.0  67982.182079  67998.505552
2026-02-26 08:46:00+00:00  67996.5  67985.363839  67998.387578
✅ Signal detection ready.
Position: None (flat — ready to trade)
✅ Order execution ready.
✅ Risk management ready.
✅ Logging ready.

🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀
   BTCUSD BOT — Delta Exchange Testnet
   Strategy: EMA 8/33 Pure Crossover
   TP: +3%  |  SL: -1%
   Entry: Crossover → Open + B

KeyboardInterrupt: 